# 4C. Keras-only CBOW - document3.txt
Same approach as 4B, but tiny corpus. We use `window=1` since sentences are short,
and train for many epochs because there are few training pairs.

In [1]:
import numpy as np
import re
from tensorflow import keras
from tensorflow.keras import layers

In [2]:
with open("../../datasets/CBOW/document3.txt") as f:
    text = f.read()

sentences = []
for raw in text.split("."):
    if raw.strip():
        tokens = re.findall(r"[a-z]+", raw.lower())
        if len(tokens) >= 3:
            sentences.append(tokens)

print("sentences:", len(sentences))

sentences: 9


In [3]:
vocab = {}
for sent in sentences:
    for word in sent:
        if word not in vocab:
            vocab[word] = len(vocab)
vocab_size = len(vocab)
print("vocab:", vocab_size)

vocab: 93


In [4]:
window = 1
contexts = []
targets = []
for sent in sentences:
    ids = [vocab[w] for w in sent]
    for i in range(window, len(ids) - window):
        context = ids[i-window:i] + ids[i+1:i+window+1]
        contexts.append(context)
        targets.append(ids[i])

contexts = np.array(contexts)
targets = np.array(targets)
print("pairs:", contexts.shape)

pairs: (161, 2)


In [5]:
model = keras.Sequential([
    layers.Input(shape=(2*window,)),
    layers.Embedding(vocab_size, 50, name="embed"),
    layers.GlobalAveragePooling1D(),
    layers.Dense(vocab_size, activation="softmax"),
])
model.compile(optimizer="adam", loss="sparse_categorical_crossentropy")
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embed (Embedding)               │ (None, 2, 50)          │         4,650 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling1d        │ (None, 50)             │             0 │
│ (GlobalAveragePooling1D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 93)             │         4,743 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 9,393 (36.69 KB)

 Trainable params: 9,393 (36.69 KB)

 Non-trainable params: 0 (0.00 B)

In [6]:
history = model.fit(contexts, targets, epochs=100)

Epoch 1/100
1/6 ━━━━━━━━━━━━━━━━━━━━ 1s 256ms/step - loss: 4.5337
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 4.5341
Epoch 2/100
1/6 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - loss: 4.5182
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 4.5207
Epoch 3/100
1/6 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - loss: 4.5077
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 4.5092
Epoch 4/100
1/6 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - loss: 4.4997
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 4.4978
Epoch 5/100
1/6 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - loss: 4.4851
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 4.4863
Epoch 6/100
1/6 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - loss: 4.4798
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 4.4746
Epoch 7/100
1/6 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - loss: 4.4741
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 4.4631
Epoch 8/100
1/6 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - loss: 4.4470
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 4.4516
Epoch 9/100
1/6 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - loss: 4.4398
6/6 ━━

In [7]:
embeddings = model.get_layer("embed").get_weights()[0]
norms = np.linalg.norm(embeddings, axis=1, keepdims=True)
embeddings_norm = embeddings / (norms + 1e-9)

id_to_word = {i: w for w, i in vocab.items()}

for word in ["covid", "virus", "influenza", "transmission"]:
    if word in vocab:
        i = vocab[word]
        sims = embeddings_norm @ embeddings_norm[i]
        top = sims.argsort()[::-1][1:6]
        print(f"\nsimilar to {word!r}:")
        for j in top:
            print(f"  {id_to_word[j]}: {sims[j]:.3f}")


similar to 'covid':
  higher: 0.829
  influenza: 0.609
  hours: 0.511
  estimates: 0.434
  has: 0.433

similar to 'virus':
  for: 0.581
  and: 0.559
  two: 0.539
  interval: 0.531
  both: 0.520



similar to 'influenza':
  estimates: 0.756
  covid: 0.609
  both: 0.564
  shed: 0.404
  while: 0.394

similar to 'transmission':
  symptoms: 0.820
  speed: 0.786
  secondary: 0.677
  driver: 0.669
  appearance: 0.629
